In [10]:
from __future__ import annotations
from pathlib import Path
import hashlib
import json
import math
import random
import copy
import platform
import warnings
import os

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from numpy.lib.stride_tricks import sliding_window_view

# ===== 1. CẤU HÌNH — thay ROOT theo thư mục chứa file trên máy bạn =====
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
RUN_AE_REFITS = True  # Lần chạy đầy đủ tốn nhiều thời gian
OUTPUT_DIR = ROOT / 'outputs' / (
    '06_4_joint_behavior_A_full' if RUN_AE_REFITS else '06_4_joint_behavior_A_quick'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def existing_file(*paths):
    return next((p for p in paths if p.is_file()), paths[0])

DEV_PATH = existing_file(ROOT/'03_model_ready_development.csv',
                         ROOT/'outputs/03_feature_engineering/03_model_ready_development.csv')
FULL_FEATURES_PATH = existing_file(ROOT/'03_features_all.csv',
                                   ROOT/'outputs/03_feature_engineering/03_features_all.csv')  # Lọc Val-select
SOURCE_PATH = existing_file(ROOT/'02_ohlcv_with_split.csv',
                            ROOT/'outputs/02_time_split_70_15_15/02_ohlcv_with_split.csv')  # Lọc Train+Val-select
FEATURE_CONFIG_PATH = existing_file(ROOT/'03_feature_config.json',
                                   ROOT/'outputs/03_feature_engineering/03_feature_config.json')
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
RAW_NAMES = ['gap','intraday','range','volume']
RAW_COLS = ['raw_gap','raw_intraday','raw_range','raw_log_volume']
OHLCV = ['open','high','low','close','volume']
EXPECTED_SHA_DEV = '0f035c54869b7389548ce6ce0b574335ed149e8378b09351eaa8245b840f472a'
W_BASE = 60
MAD_FACTOR = 1.4826
FLOOR_Q = 0.10
K_LOF = 20
IF_TREES = 200
IF_SAMPLES = 256
IF_SEED = 42
SAMPLE_PER_TICKER_MONTH = 2
SYNTH_SEED = 2026
W_FLOOR_GRID = [(20,1.0),(60,0.5),(60,1.0),(60,2.0),(120,1.0)]
AE_EPOCHS = 1500
AE_LR = 0.001
AE_PATIENCE = 80
AE_BATCH = 512

# Chỉ dùng checkpoint AE của Bước 06.2 hoặc bản tái lập kiểm chứng, TUYỆT ĐỐI KHÔNG tự chuyển sang Bước 05.
AE_CANDIDATES = [
    ROOT / '06_2_ae_seed42_lr001_epoch1495_best.pt',
    ROOT / 'outputs/06_2_ae_validation_A/06_2_ae_seed42_lr001_epoch1495_best.pt',
    *sorted((ROOT/'outputs/06_2_ae_validation_A').glob('06_2_ae_seed42_lr001_epoch*_best.pt')),
    *sorted(ROOT.glob('06_2_ae_seed42_lr001_epoch*_best.pt')),
    ROOT / '06_4_ae_ref_reproduced_seed42_lr001.pt',
]
IF_CANDIDATES = [ROOT/'05_if.joblib', ROOT/'outputs/05_model_training_A/05_if.joblib']
LOF_CANDIDATES = [
    ROOT/'06_3_lof_baseline_k20_euclidean.joblib',
    ROOT/'outputs/06_3_lof_validation_A/06_3_lof_baseline_k20_euclidean.joblib',
    ROOT/'05_lof.joblib',
]


def find_one(candidates, label):
    result = next((p for p in candidates if p.is_file()), None)
    if result is None:
        raise FileNotFoundError(f'Không tìm thấy {label}. Các đường dẫn đã kiểm tra:\n' +
                                '\n'.join(str(p) for p in candidates))
    return result


def sha256_file(path):
    hasher=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            hasher.update(block)
    return hasher.hexdigest()


def as_float32(df):
    return df[FEATURES].to_numpy(dtype=np.float32)


def rank_stats(a,b):
    a=np.asarray(a,dtype=float);b=np.asarray(b,dtype=float)
    assert len(a)==len(b) and len(a)>10
    out={'spearman':float(spearmanr(a,b).statistic)}
    for rate in [0.01,0.05]:
        n=int(np.ceil(len(a)*rate))
        ia=set(np.argsort(-a,kind='stable')[:n]);ib=set(np.argsort(-b,kind='stable')[:n])
        out[f'top{int(100*rate)}_overlap_pct']=100*len(ia&ib)/n
    return out

In [11]:
assert DEV_PATH.is_file() and FULL_FEATURES_PATH.is_file() and SOURCE_PATH.is_file()
assert FEATURE_CONFIG_PATH.is_file()
if sha256_file(DEV_PATH) != EXPECTED_SHA_DEV:
    raise ValueError('03_model_ready_development.csv khác phiên bản đã dùng tại Bước 05–06.3')

with FEATURE_CONFIG_PATH.open(encoding='utf-8') as f:
    fconf=json.load(f)
assert fconf['window_prior_valid_feature_values_per_ticker']==60
assert fconf['train_floor_quantile_positive_mad_scales']==0.1
floors_base={x:float(fconf['train_floors'][x]) for x in RAW_NAMES}

# File Development KHÔNG chứa Test, nhưng có Val-calibrate: lọc trước mọi phân tích.
dev_all=pd.read_csv(DEV_PATH,encoding='utf-8-sig')
assert not dev_all.duplicated(['ticker','date']).any()
assert set(dev_all.split)=={'train','val'}
allowed=dev_all.split.eq('train') | (dev_all.split.eq('val') & dev_all.val_role.eq('select'))
dev=dev_all.loc[allowed,['ticker','date','split','val_role',*FEATURES]].copy()
del dev_all
train=dev.loc[dev.split.eq('train')].sort_values(['date','ticker']).reset_index(drop=True)
val=dev.loc[dev.split.eq('val')].sort_values(['date','ticker']).reset_index(drop=True)
assert len(train)==10387 and len(val)==1430
assert pd.to_datetime(train.date).max()<pd.to_datetime(val.date).min()
X_train=as_float32(train);X_val=as_float32(val)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()
print('Bước 06.4:',len(train),'Train;',len(val),'Val-select. Không sử dụng Val-calibrate/Test.')

Bước 06.4: 10387 Train; 1430 Val-select. Không sử dụng Val-calibrate/Test.


In [12]:
torch.set_num_threads(1)
class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(4,2),nn.Tanh(),nn.Linear(2,4))
    def forward(self,x): return self.net(x)

@torch.no_grad()
def ae_score(model,x):
    model.eval()
    t = torch.from_numpy(
        np.array(x, dtype=np.float32, copy=True)
    )
    return (model(t)-t).square().mean(dim=1).cpu().numpy().astype(float)

if_path=find_one(IF_CANDIDATES,'IF Bước 05')
lof_path=find_one(LOF_CANDIDATES,'LOF k20 Bước 06.3')
ae_path=find_one(AE_CANDIDATES,'AE checkpoint Bước 06.2')
with warnings.catch_warnings():
    warnings.simplefilter('ignore',category=UserWarning)
    model_if=joblib.load(if_path)
    model_lof=joblib.load(lof_path)
ckpt=torch.load(ae_path,map_location='cpu',weights_only=True)
assert ckpt.get('seed')==42 and ckpt.get('best_epoch',0)>=1400, 'AE không phải checkpoint Bước 06.2'
assert list(ckpt['features'])==FEATURES
model_ae=SmallAE()
model_ae.load_state_dict(ckpt['state_dict'])
model_ae.eval()
models={'IF':model_if,'AE':model_ae,'LOF':model_lof}

def score_all(x):
    x=np.asarray(x,dtype=np.float32)
    return {
        'IF':-model_if.score_samples(x),
        'AE':ae_score(model_ae,x),
        'LOF':-model_lof.score_samples(x),
    }

baseline_scores=score_all(X_val)
# Khớp dữ liệu Validation 06.1/06.2/06.3 theo KEY, KHÔNG dựa vào thứ tự CSV.
reference_files={
    'IF':([ROOT/'06_1_baseline_val_scores.csv',
           ROOT/'outputs/06_1_if_validation_A/06_1_baseline_val_scores.csv'], 'score_if'),
    'AE':([ROOT/'06_2_ae_val_select_scores_seed42.csv',
           ROOT/'06_2_ae_val_select_scores_seed42(1).csv',
           ROOT/'outputs/06_2_ae_validation_A/06_2_ae_val_select_scores_seed42.csv'], 'score_ae'),
    'LOF':([ROOT/'06_3_lof_baseline_val_scores.csv',
            ROOT/'outputs/06_3_lof_validation_A/06_3_lof_baseline_val_scores.csv'], 'score_lof'),
}
reproduction=[]
for m,(candidates,col) in reference_files.items():
    p=find_one(candidates, f'điểm tham chiếu {m}')
    old=pd.read_csv(p,encoding='utf-8-sig')
    table=val[['ticker','date']].merge(old[['ticker','date',col]],on=['ticker','date'],how='left',validate='one_to_one')
    assert table[col].notna().all()
    diff=np.abs(baseline_scores[m]-table[col].to_numpy(dtype=float))
    limit={'IF':1e-5,'AE':5e-4,'LOF':1e-5}[m]
    if diff.max()>limit:
        raise ValueError(f'{m} không khớp checkpoint và bảng điểm cũ; max diff={diff.max():.6g}. '
                         'Không được dùng checkpoint khác cho cùng thí nghiệm.')
    reproduction.append({'model':m,'max_abs_score_diff':float(diff.max()),'tolerance':limit,
                        'model_file':str({'IF':if_path,'AE':ae_path,'LOF':lof_path}[m])})
print('Đối chiếu điểm Bước 06.1–06.3:')
print(pd.DataFrame(reproduction).to_string(index=False))
pd.DataFrame(reproduction).to_csv(OUTPUT_DIR/'06_4_checkpoint_reproduction.csv',index=False)

Đối chiếu điểm Bước 06.1–06.3:
model  max_abs_score_diff  tolerance                                                                                                                   model_file
   IF                 0.0    0.00001                             d:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\05_model_training_A\05_if.joblib
   AE                 0.0    0.00050  d:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_2_ae_validation_A\06_2_ae_seed42_lr001_epoch1495_best.pt
  LOF                 0.0    0.00001 d:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_3_lof_validation_A\06_3_lof_baseline_k20_euclidean.joblib


In [13]:
usecols=['ticker','date','split','val_role',*OHLCV,'prev_valid_close']
for name in RAW_NAMES:
    usecols += [f'hist_median_{name}',f'hist_mad_{name}']
# Đọc và LỌC NGAY; không dùng điểm hay đặc trưng của Val-calibrate/Test để chọn.
all_features=pd.read_csv(FULL_FEATURES_PATH,usecols=usecols,encoding='utf-8-sig')
all_features=all_features.loc[all_features.split.eq('val') &
                              all_features.val_role.eq('select')].copy()
raw_val=val[['ticker','date',*FEATURES]].merge(all_features.drop(columns=['split','val_role']),
                                  on=['ticker','date'],how='left',validate='one_to_one')
del all_features
assert len(raw_val)==1430 and raw_val[usecols[4:]].notna().all().all()
for name in RAW_NAMES:
    np.testing.assert_allclose(raw_val[f'z_{name}'].to_numpy(),
                               val[f'z_{name}'].to_numpy(),rtol=0,atol=1e-10)

raw_val['month']=raw_val['date'].str.slice(0,7)
# Mỗi tháng của từng mã lấy 2 phiên bằng seed cố định, KHÔNG dựa vào điểm mô hình.
samples=pd.concat([g.sample(n=min(SAMPLE_PER_TICKER_MONTH,len(g)), random_state=SYNTH_SEED)
                   for _,g in raw_val.groupby(['ticker','month'],sort=True)],ignore_index=True)
samples=samples.sort_values(['date','ticker']).reset_index(drop=True)
assert samples[['ticker','date']].duplicated().sum()==0

raw_matrix=samples[OHLCV].to_numpy(dtype=float)
prev_close=samples['prev_valid_close'].to_numpy(dtype=float)
hist_m=np.column_stack([samples[f'hist_median_{name}'].to_numpy(float) for name in RAW_NAMES])
hist_mad=np.column_stack([samples[f'hist_mad_{name}'].to_numpy(float) for name in RAW_NAMES])
floors=np.array([floors_base[name] for name in RAW_NAMES])

def raw_from_ohlcv(v,close_prev):
    O,H,L,C,V=np.asarray(v,dtype=float).T
    return np.column_stack([
        np.log(O/close_prev),np.log(C/O),np.log(H/L),np.log1p(V)
    ])

def check_ohlcv(v):
    O,H,L,C,V=np.asarray(v,dtype=float).T
    return bool(np.isfinite(v).all() and (O>0).all() and (L>0).all() and
                (H>0).all() and (C>0).all() and (V>=0).all() and
                (H>=np.maximum.reduce([O,L,C])).all() and
                (L<=np.minimum.reduce([O,H,C])).all())

def features_from_ohlcv(v):
    raw=raw_from_ohlcv(v,prev_close)
    scale=np.maximum(MAD_FACTOR*hist_mad,floors)
    return raw, (raw-hist_m)/scale

assert check_ohlcv(raw_matrix)
raw_original,z_original=features_from_ohlcv(raw_matrix)
np.testing.assert_allclose(z_original,samples[FEATURES].to_numpy(),atol=1e-9,rtol=0)


def create_mutant(base,scenario):
    m=base.copy()
    if scenario in ['gap_amplify','combination']:
        gap_z=samples['z_gap'].to_numpy()
        # Điều chỉnh gap theo chiều làm tăng |z_gap|, giữ intraday và range y nguyên.
        direction=np.where(gap_z>=0,1.0,-1.0)
        factor=np.exp(0.05*direction)
        m[:,:4] *= factor[:,None]
    if scenario=='intraday_close_shift':
        # Chỉ thay Close trong [Low, High] — đảm bảo không sửa Gap/Range/Volume.
        o,h,l,c,v=m.T
        lower=l+0.05*(h-l);upper=l+0.95*(h-l)
        median_intra=samples['hist_median_intraday'].to_numpy(float)
        lower_dist=np.abs(np.log(lower/o)-median_intra)
        upper_dist=np.abs(np.log(upper/o)-median_intra)
        m[:,3]=np.where(lower_dist>=upper_dist,lower,upper)
    if scenario in ['range_wider','combination']:
        m[:,1] *= 1.05  # high
        m[:,2] /= 1.05  # low
    if scenario in ['volume_x4','combination']:
        m[:,4] *= 4.0
    if scenario=='volume_quarter':
        m[:,4] = np.maximum(0,np.rint(m[:,4]/4.0))
    return m

SCENARIOS=[
    ('gap_amplify',[0]),
    ('intraday_close_shift',[1]),
    ('range_wider',[2]),
    ('volume_x4',[3]),
    ('volume_quarter',[3]),
    ('combination',[0,2,3]),
]

synthetic_tables=[]
for scenario,expected_changed in SCENARIOS:
    mut=create_mutant(raw_matrix,scenario)
    assert check_ohlcv(mut),f'Biến đổi {scenario} làm sai logic OHLCV'
    raw_new,z_new=features_from_ohlcv(mut)
    assert np.isfinite(z_new).all()
    # Nếu chỉ đổi 1 thành phần, các thành phần không mục tiêu phải giữ nguyên.
    invariant_cols=[j for j in range(4) if j not in expected_changed]
    np.testing.assert_allclose(raw_new[:,invariant_cols],raw_original[:,invariant_cols],
                               atol=1e-11,rtol=0)
    # Không tính lại nền W hoặc fit mô hình cho một phản thực tại phiên hiện tại.
    snew=score_all(z_new)
    table=samples[['ticker','date','month']].copy()
    table['scenario']=scenario
    table['is_effective']=(np.max(np.abs(z_new-z_original),axis=1)>1e-8)
    table['target_z_increased_abs']=np.any(np.abs(z_new[:,expected_changed])>
                                           np.abs(z_original[:,expected_changed])+1e-9,axis=1)
    for j,name in enumerate(RAW_NAMES):
        table[f'original_z_{name}']=z_original[:,j]
        table[f'mutated_z_{name}']=z_new[:,j]
    for model in ['IF','AE','LOF']:
        # Điểm gốc phải là điểm theo cùng model, cùng hàng, trước khi thay OHLCV.
        # căn chỉnh bằng khóa rõ ràng, không sử dụng thứ tự merge ngầm
        source_score=val[['ticker','date']].copy()
        source_score['score']=baseline_scores[model]
        aligned=samples[['ticker','date']].merge(source_score,on=['ticker','date'],how='left',validate='one_to_one')
        orig=aligned['score'].to_numpy(float)
        new=snew[model]
        reference=np.sort(baseline_scores[model])
        original_pct=100*np.searchsorted(reference,orig,side='right')/len(reference)
        new_pct=100*np.searchsorted(reference,new,side='right')/len(reference)
        table[f'original_{model}']=orig
        table[f'mutated_{model}']=new
        table[f'delta_score_{model}']=new-orig
        table[f'delta_percentile_pts_{model}']=new_pct-original_pct
    synthetic_tables.append(table)

synthetic=pd.concat(synthetic_tables,ignore_index=True)
synthetic.to_csv(OUTPUT_DIR/'06_4_synthetic_pairs.csv',index=False,encoding='utf-8-sig')

summary_rows=[]
for scenario,model,g in ((s,m,synthetic[synthetic.scenario.eq(s)])
                           for s,_ in SCENARIOS for m in ['IF','AE','LOF']):
    d=g.loc[g.is_effective,f'delta_score_{model}'].to_numpy(dtype=float)
    dp=g.loc[g.is_effective,f'delta_percentile_pts_{model}'].to_numpy(dtype=float)
    amp=g.loc[g.is_effective & g.target_z_increased_abs,f'delta_score_{model}'].to_numpy(dtype=float)
    summary_rows.append({
        'scenario':scenario,'model':model,'n_cases':len(g),
        'n_effective':len(d),
        'n_with_larger_target_abs_z':int(g.target_z_increased_abs.sum()),
        'score_increased_pct':float(100*np.mean(d>0)) if len(d) else np.nan,
        'score_increased_among_amplified_pct':float(100*np.mean(amp>0)) if len(amp) else np.nan,
        'median_delta_score':float(np.median(d)) if len(d) else np.nan,
        'median_delta_percentile_pts':float(np.median(dp)) if len(dp) else np.nan,
    })
synthetic_summary=pd.DataFrame(summary_rows)
synthetic_summary.to_csv(OUTPUT_DIR/'06_4_synthetic_summary.csv',index=False,encoding='utf-8-sig')
print('\nPHẢN ỨNG VỚI OHLCV GIẢ LẬP (chưa phải accuracy):')
print(synthetic_summary[['scenario','model','n_effective','score_increased_pct',
                        'median_delta_percentile_pts']].round(2).to_string(index=False))


PHẢN ỨNG VỚI OHLCV GIẢ LẬP (chưa phải accuracy):
            scenario model  n_effective  score_increased_pct  median_delta_percentile_pts
         gap_amplify    IF          156               100.00                        35.38
         gap_amplify    AE          156                91.03                        27.45
         gap_amplify   LOF          156                96.15                        41.96
intraday_close_shift    IF          156                82.69                         5.17
intraday_close_shift    AE          156                75.64                         8.11
intraday_close_shift   LOF          156                64.10                         4.09
         range_wider    IF          156               100.00                        42.06
         range_wider    AE          156               100.00                        45.56
         range_wider   LOF          156               100.00                        50.31
           volume_x4    IF          156           

In [14]:
# Lọc từ dữ liệu 02 trước khi tạo rolling; KHÔNG tính đặc trưng cho Val-calibrate/Test.
base_data=pd.read_csv(SOURCE_PATH,encoding='utf-8-sig')
base_data=base_data.loc[base_data.split.eq('train') |
                        (base_data.split.eq('val') & base_data.val_role.eq('select')),
                        ['ticker','date',*OHLCV,'split','val_role']].copy()
base_data=base_data.sort_values(['ticker','date']).reset_index(drop=True)
assert not base_data.duplicated(['ticker','date']).any()
assert base_data['ticker'].nunique()==13
assert (base_data[OHLCV[:4]]>0).all().all() and (base_data.volume>=0).all()


def recalc_for_w(w):
    d=base_data.copy()
    d['prev']=d.groupby('ticker',sort=False)['close'].shift(1)
    d['raw_gap']=np.log(d.open/d.prev)
    d['raw_intraday']=np.log(d.close/d.open)
    d['raw_range']=np.log(d.high/d.low)
    d['raw_log_volume']=np.log1p(d.volume)
    for name in RAW_NAMES:
        d[f'median_{name}']=np.nan
        d[f'mad_{name}']=np.nan
    for _,g in d.groupby('ticker',sort=False):
        ii=g.index
        for name,raw in zip(RAW_NAMES,RAW_COLS):
            x=g[raw].to_numpy(float); med=np.full(len(x),np.nan);mad=np.full(len(x),np.nan)
            if len(x)>w:
                windows=sliding_window_view(x,w)[:-1]
                ok=np.isfinite(windows).all(axis=1)
                m=np.median(windows[ok],axis=1)
                med2=med[w:];mad2=mad[w:]
                med2[ok]=m
                mad2[ok]=np.median(np.abs(windows[ok]-m[:,None]),axis=1)
            d.loc[ii,f'median_{name}']=med
            d.loc[ii,f'mad_{name}']=mad
    enough=np.isfinite(d[RAW_COLS+[f'median_{n}' for n in RAW_NAMES]+[
        f'mad_{n}' for n in RAW_NAMES]].to_numpy(float)).all(axis=1)
    candidate=enough & d.split.eq('train').to_numpy()
    learned={}
    for name in RAW_NAMES:
        scale=MAD_FACTOR*d[f'mad_{name}'].to_numpy(float)
        positive=scale[candidate & (scale>0)]
        if len(positive)==0:raise ValueError(f'W={w} thiếu MAD dương của {name}')
        learned[name]=float(np.quantile(positive,FLOOR_Q))
    return d,enough,learned


def apply_floor(d,enough,base_floors,multiplier):
    d=d.copy()
    for name,raw in zip(RAW_NAMES,RAW_COLS):
        denom=np.maximum(MAD_FACTOR*d[f'mad_{name}'].to_numpy(float),base_floors[name]*multiplier)
        d[f'z_{name}']=(d[raw].to_numpy(float)-d[f'median_{name}'].to_numpy(float))/denom
    d=d.loc[enough,['ticker','date','split',*FEATURES]].copy()
    assert np.isfinite(d[FEATURES].to_numpy(float)).all()
    return d


def fit_ae_for_case(xtr,xv,seed=42):
    """Cùng kiến trúc và giao thức Step 06.2; rất tốn thời gian, chạy khi bật công tắc."""
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)
    xt=torch.from_numpy(xtr.astype(np.float32));vv=torch.from_numpy(xv.astype(np.float32))
    model=SmallAE();optimizer=torch.optim.Adam(model.parameters(),lr=AE_LR)
    loader=DataLoader(TensorDataset(xt),batch_size=AE_BATCH,shuffle=True,
                      generator=torch.Generator().manual_seed(seed),num_workers=0)
    criterion=nn.MSELoss();best=float('inf');best_state=None;stale=0
    for epoch in range(1,AE_EPOCHS+1):
        model.train()
        for (batch,) in loader:
            optimizer.zero_grad(set_to_none=True)
            loss=criterion(model(batch),batch)
            loss.backward();optimizer.step()
        model.eval()
        with torch.no_grad():current=float(criterion(model(vv),vv).item())
        if current < best-1e-5:
            best=current;best_state=copy.deepcopy(model.state_dict());stale=0
        else:stale+=1
        if stale>=AE_PATIENCE:break
    model.load_state_dict(best_state)
    return ae_score(model,xv)

cached_w={}
for w,_ in W_FLOOR_GRID:
    if w not in cached_w:cached_w[w]=recalc_for_w(w)

# Phải tái tạo đúng dữ liệu W=60, floor=1 trước khi so sánh các W.
d60,ready60,floor60=cached_w[60]
for key in RAW_NAMES:
    np.testing.assert_allclose(floor60[key],floors_base[key],rtol=0,atol=1e-12)
base_rebuilt=apply_floor(d60,ready60,floor60,1.0)
match=val[['ticker','date',*FEATURES]].merge(
    base_rebuilt.loc[base_rebuilt.split.eq('val')].drop(columns='split'),
    on=['ticker','date'],suffixes=('_old','_new'),how='left',validate='one_to_one')
for name in FEATURES:
    np.testing.assert_allclose(match[f'{name}_old'],match[f'{name}_new'],rtol=0,atol=1e-9)

sensitivity_scores=[]
sensitivity_rows=[]
for w,mult in W_FLOOR_GRID:
    d,ready,learned=cached_w[w]
    prepared=apply_floor(d,ready,learned,mult)
    tr=prepared.loc[prepared.split.eq('train')].sort_values(['date','ticker'])
    vv=prepared.loc[prepared.split.eq('val')].sort_values(['date','ticker'])
    assert len(vv)==len(val)
    assert (vv[['ticker','date']].to_numpy()==val[['ticker','date']].to_numpy()).all()
    xtr=tr[FEATURES].to_numpy(np.float32);xv=vv[FEATURES].to_numpy(np.float32)
    model_i=IsolationForest(n_estimators=IF_TREES,max_samples=IF_SAMPLES,max_features=1.0,
                            contamination='auto',random_state=IF_SEED,n_jobs=1)
    model_i.fit(xtr)
    model_l=LocalOutlierFactor(n_neighbors=K_LOF,metric='minkowski',p=2,
                              novelty=True,contamination='auto',n_jobs=1)
    model_l.fit(xtr)
    ss={'IF':-model_i.score_samples(xv), 'LOF':-model_l.score_samples(xv)}
    if RUN_AE_REFITS:
        ss['AE']=fit_ae_for_case(xtr,xv)
        if w==60 and mult==1.0:
            np.testing.assert_allclose(ss['AE'],baseline_scores['AE'],rtol=0,atol=5e-4)
    label=f'W{w}_floorx{mult:g}'
    for mod,arr in ss.items():
        metrics=rank_stats(baseline_scores[mod],arr)
        sensitivity_rows.append({'case':label,'W':w,'floor_multiplier':mult,
                                 'model':mod,'n_train':len(tr),'n_val_select':len(vv),**metrics})
        temp=vv[['ticker','date']].copy()
        temp['case']=label;temp['model']=mod;temp['score']=arr
        sensitivity_scores.append(temp)
    print('Đã refit',label,'Train',len(tr),'Val',len(vv),'Models',list(ss),flush=True)

sensitivity=pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUTPUT_DIR/'06_4_w_floor_sensitivity.csv',index=False,encoding='utf-8-sig')
pd.concat(sensitivity_scores,ignore_index=True).to_csv(
    OUTPUT_DIR/'06_4_w_floor_scores.csv',index=False,encoding='utf-8-sig')
print('\nKIỂM TRA W VÀ MỨC SÀN (không phải chọn cấu hình tối ưu):')
print(sensitivity.round(3).to_string(index=False))

Đã refit W20_floorx1 Train 10907 Val 1430 Models ['IF', 'LOF', 'AE']
Đã refit W60_floorx0.5 Train 10387 Val 1430 Models ['IF', 'LOF', 'AE']
Đã refit W60_floorx1 Train 10387 Val 1430 Models ['IF', 'LOF', 'AE']
Đã refit W60_floorx2 Train 10387 Val 1430 Models ['IF', 'LOF', 'AE']
Đã refit W120_floorx1 Train 9607 Val 1430 Models ['IF', 'LOF', 'AE']

KIỂM TRA W VÀ MỨC SÀN (không phải chọn cấu hình tối ưu):
         case   W  floor_multiplier model  n_train  n_val_select  spearman  top1_overlap_pct  top5_overlap_pct
  W20_floorx1  20               1.0    IF    10907          1430     0.793            66.667            69.444
  W20_floorx1  20               1.0   LOF    10907          1430     0.533            46.667            56.944
  W20_floorx1  20               1.0    AE    10907          1430     0.820            60.000            63.889
W60_floorx0.5  60               0.5    IF    10387          1430     0.979            80.000            91.667
W60_floorx0.5  60               0.5   LO

In [19]:

# ============================================================
# BƯỚC 06.4 - LƯU KẾT QUẢ FULL: IF + AE + LOF
# KHÔNG HUẤN LUYỆN LẠI MÔ HÌNH
# ============================================================

from pathlib import Path
import pandas as pd
import json

# 1. KIỂM TRA KẾT QUẢ TRUE ĐÃ CÓ TRONG BỘ NHỚ

assert "sensitivity" in globals(), (
    "Chưa có bảng sensitivity."
)

assert "sensitivity_scores" in globals(), (
    "Chưa có bảng điểm sensitivity_scores."
)

assert len(sensitivity) == 15, (
    f"Cần 15 kết quả, hiện có {len(sensitivity)}."
)

assert set(sensitivity["model"]) == {"IF", "AE", "LOF"}, (
    "Chưa có đủ IF, AE và LOF."
)

# Ghép điểm của 5 cấu hình × 3 mô hình
scores_full = pd.concat(
    sensitivity_scores,
    ignore_index=True
)

expected_rows = 5 * 3 * len(val)

assert len(scores_full) == expected_rows, (
    f"Cần {expected_rows} dòng điểm, "
    f"nhưng hiện có {len(scores_full)}."
)

# Mỗi cấu hình - mô hình phải có đủ phiên Validation
counts = scores_full.groupby(["case", "model"]).size()

assert len(counts) == 15
assert counts.eq(len(val)).all()

print("PASS: Có đủ 5 cấu hình × 3 mô hình.")
print("Số dòng tổng hợp:", len(sensitivity))
print("Số dòng điểm:", len(scores_full))


# 2. CHỈ ĐỊNH THƯ MỤC KẾT QUẢ TRUE

RUN_AE_REFITS = True

OUTPUT_DIR = (
    Path(ROOT)
    / "outputs"
    / "06_4_joint_behavior_A_full"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nThư mục lưu kết quả mới:")
print(OUTPUT_DIR.resolve())


# 3. LƯU CÁC FILE CSV MỚI

sensitivity.to_csv(
    OUTPUT_DIR / "06_4_w_floor_sensitivity.csv",
    index=False,
    encoding="utf-8-sig"
)

scores_full.to_csv(
    OUTPUT_DIR / "06_4_w_floor_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

synthetic.to_csv(
    OUTPUT_DIR / "06_4_synthetic_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

synthetic_summary.to_csv(
    OUTPUT_DIR / "06_4_synthetic_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(reproduction).to_csv(
    OUTPUT_DIR / "06_4_checkpoint_reproduction.csv",
    index=False,
    encoding="utf-8-sig"
)


# 4. LƯU CONFIG JSON - XÁC NHẬN TRUE

qa = {
    "train": int(len(train)),
    "val_select": int(len(val)),
    "n_synthetic_base_sessions": int(len(samples)),
    "n_synthetic_modified_sessions": int(len(synthetic)),
    "n_invalid_synthetic_ohlcv": 0,
    "n_synthetic_scenarios": len(SCENARIOS),
    "n_w_floor_cases": len(W_FLOOR_GRID),

    "ae_checkpoint": str(ae_path),
    "ae_checkpoint_epoch": int(ckpt["best_epoch"]),

    "ae_refit_in_w_floor": True,

    "no_test_or_calibrate_used": True,
    "warning_no_official_HOSE_band_or_tick_check": True,
    "no_proxy_no_alert_threshold": True,

    "input_sha256": sha256_file(DEV_PATH),

    "baseline_model_files": {
        "IF": str(if_path),
        "AE": str(ae_path),
        "LOF": str(lof_path)
    }
}

with (
    OUTPUT_DIR / "06_4_run_config.json"
).open("w", encoding="utf-8") as f:
    json.dump(qa, f, ensure_ascii=False, indent=2)


# 5. TẠO BÁO CÁO MỚI - CÓ ĐỦ BA MÔ HÌNH

lines = [
    "# Bước 06.4 — Kiểm tra hành vi chung (phương án A)",
    "",
    "IF là mô hình chính; AE và LOF là mô hình so sánh.",
    "",
    f"Train: {len(train):,} phiên.",
    f"Validation chọn cấu hình: {len(val):,} phiên.",
    "",
    "## Đối chiếu checkpoint",
    pd.DataFrame(reproduction).to_string(index=False),
    "",
    "## Biến đổi OHLCV có kiểm soát",
    f"Số phiên gốc: {len(samples)}.",
    f"Số phiên biến đổi: {len(synthetic)}.",
    "",
    synthetic_summary.round(3).to_string(index=False),
    "",
    "## Độ nhạy W / mức sàn MAD",
    "",
    "Đã tính lại đặc trưng và huấn luyện lại "
    "IF, AE, LOF cho cả 5 cấu hình.",
    "",
    "Mức sàn được học từ Train.",
    "Không sử dụng Test hoặc Validation hiệu chỉnh.",
    "",
    sensitivity.round(3).to_string(index=False),
    "",
    "## Giới hạn nghiên cứu",
    "- Không có nhãn bất thường thật.",
    "- Không suy ra Accuracy hoặc Recall từ tỷ lệ tăng điểm.",
    "- Top 1% và Top 5% chỉ phục vụ phân tích.",
    "- Số phiên Train khác nhau khi thay W.",
    "- Phép biến đổi chưa kiểm tra quy tắc trần/sàn HOSE.",
    "- Chưa lựa chọn cấu hình tối ưu.",
    "- Chưa sử dụng Test hoặc Validation hiệu chỉnh.",
    "",
    "Bước tiếp theo: 06.5 — So sánh tổng hợp IF, AE và LOF."
]

(OUTPUT_DIR / "06_4_joint_behavior_report.md").write_text(
    "\n".join(lines),
    encoding="utf-8"
)


# 6. ĐỌC LẠI FILE ĐỂ KIỂM TRA

saved_summary = pd.read_csv(
    OUTPUT_DIR / "06_4_w_floor_sensitivity.csv"
)

saved_scores = pd.read_csv(
    OUTPUT_DIR / "06_4_w_floor_scores.csv"
)

with (
    OUTPUT_DIR / "06_4_run_config.json"
).open(encoding="utf-8") as f:
    saved_config = json.load(f)

assert len(saved_summary) == 15
assert len(saved_scores) == expected_rows
assert saved_config["ae_refit_in_w_floor"] is True

assert set(saved_summary["model"]) == {
    "IF", "AE", "LOF"
}

print("\n====================================")
print("PASS: ĐÃ LƯU KẾT QUẢ TRUE THÀNH CÔNG")
print("====================================")
print("Thư mục:", OUTPUT_DIR.resolve())
print("AE refit:", saved_config["ae_refit_in_w_floor"])
print("Bảng tổng hợp:", len(saved_summary), "dòng")
print("Bảng điểm:", len(saved_scores), "dòng")
print("Mô hình:", sorted(saved_summary["model"].unique()))


PASS: Có đủ 5 cấu hình × 3 mô hình.
Số dòng tổng hợp: 15
Số dòng điểm: 21450

Thư mục lưu kết quả mới:
D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_4_joint_behavior_A_full

PASS: ĐÃ LƯU KẾT QUẢ TRUE THÀNH CÔNG
Thư mục: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\06_4_joint_behavior_A_full
AE refit: True
Bảng tổng hợp: 15 dòng
Bảng điểm: 21450 dòng
Mô hình: ['AE', 'IF', 'LOF']
